In [1]:
# importar bibliotecas necessárias
from pathlib import Path
import pandas as pd
import psycopg
from sqlalchemy import create_engine

In [ ]:
# # Baixar o dataset da competição Home Credit Default Risk do Kaggle usando a biblioteca kagglehub.
# import kagglehub

# # Download latest version
# path = kagglehub.competition_download('home-credit-default-risk')

# print("Path to competition files:", path)

Path to competition files: /home/oem/.cache/kagglehub/competitions/home-credit-default-risk


In [26]:
!uv add pandas sqlalchemy "psycopg[binary]"

Resolved 9 packages in 0.71ms
Checked 9 packages in 0.16ms


In [ ]:
# # Importar os arquivos CSV para o banco de dados PostgreSQL

# # Definir o caminho para os arquivos CSV e a string de conexão do banco de dados
# PASTA = Path("~/.cache/kagglehub/competitions/home-credit-default-risk").expanduser()
# DSN = "postgresql:///home_credit"
# engine = create_engine(DSN.replace("postgresql", "postgresql+psycopg", 1))
# # Dois dos CSVs não são dados: sample_submission.csv é o formato de envio da competição e HomeCredit_columns_description.csv é o dicionário de colunas
# IGNORAR = {"sample_submission.csv", "HomeCredit_columns_description.csv"}

# for csv in sorted(PASTA.glob("*.csv")):
#     if csv.name in IGNORAR:
#         continue
#     tabela = csv.stem.lower()

#     # 1. pandas infere os tipos lendo o arquivo inteiro e cria a tabela vazia
#     df = pd.read_csv(csv)
#     df.columns = df.columns.str.lower()
#     df.head(0).to_sql(tabela, engine, index=False, if_exists="replace")
#     n = len(df)
#     del df

#     # 2. COPY envia o arquivo em blocos de 1 MB pela conexão
#     with psycopg.connect(DSN) as conn, conn.cursor() as cur, open(csv) as f:
#         with cur.copy(f"COPY {tabela} FROM STDIN WITH (FORMAT csv, HEADER true)") as copy:
#             while bloco := f.read(1 << 20):
#                 copy.write(bloco)
#         cur.execute(f"SELECT count(*) FROM {tabela}")
#         assert cur.fetchone()[0] == n, f"{tabela}: contagem diferente do CSV"

#     print(f"{tabela}: {n:,} linhas")

pos_cash_balance: 10,001,358 linhas
application_test: 48,744 linhas
application_train: 307,511 linhas
bureau: 1,716,428 linhas
bureau_balance: 27,299,925 linhas
credit_card_balance: 3,840,312 linhas
installments_payments: 13,605,401 linhas
previous_application: 1,670,214 linhas


In [ ]:
# Verificar se há linhas órfãs
# esses códigos foram executados no banco de dados PostgreSQL para verificar se há linhas órfãs nas tabelas relacionadas.

SELECT count(*)
FROM bureau b
LEFT JOIN application_train a USING (sk_id_curr)
WHERE a.sk_id_curr IS NULL;

SELECT count(*)
FROM bureau_balance bb
LEFT JOIN bureau b USING (sk_id_bureau)
WHERE b.sk_id_bureau IS NULL;

SELECT count(*)
FROM credit_card_balance ccb
LEFT JOIN application_train a USING (sk_id_curr)
WHERE a.sk_id_curr IS NULL;

SELECT count(*)
FROM credit_card_balance ccb
LEFT JOIN previous_application a USING (sk_id_prev)
WHERE a.sk_id_prev IS NULL;

SELECT count(*)
FROM installments_payments ip
LEFT JOIN application_train a USING (sk_id_curr)
WHERE a.sk_id_curr IS NULL;

SELECT count(*)
FROM installments_payments ip
LEFT JOIN previous_application a USING (sk_id_prev)
WHERE a.sk_id_prev IS NULL;

SELECT count(*)
FROM previous_application pa
LEFT JOIN application_train a USING (sk_id_curr)
WHERE a.sk_id_curr IS NULL;

SELECT count(*)
FROM POS_CASH_balance pcb
LEFT JOIN previous_application pa USING (sk_id_prev)
WHERE pa.sk_id_prev IS NULL;

SELECT count(*)
FROM POS_CASH_balance pcb
LEFT JOIN application_train a USING (sk_id_curr)
WHERE a.sk_id_curr IS NULL;

In [ ]:
# esses códigos foram executados no banco de dados PostgreSQL para verificar se há linhas órfãs nas tabelas relacionadas.

ALTER TABLE bureau
ADD CONSTRAINT sk_b_application_train
FOREIGN KEY (sk_id_curr) REFERENCES application_train(sk_id_curr);

ALTER TABLE bureau_balance
ADD CONSTRAINT fk_bb_bureau
FOREIGN KEY (sk_id_bureau) REFERENCES bureau(sk_id_bureau);

ALTER TABLE previous_application
ADD CONSTRAINT fk_pa_application_train
FOREIGN KEY (sk_id_curr) REFERENCES application_train(sk_id_curr);

ALTER TABLE installments_payments
ADD CONSTRAINT fk_ip_previous_application
FOREIGN KEY (sk_id_prev) REFERENCES previous_application(sk_id_prev);

ALTER TABLE installments_payments
ADD CONSTRAINT fk_ip_application_train
FOREIGN KEY (sk_id_curr) REFERENCES application_train(sk_id_curr);

ALTER TABLE credit_card_balance
ADD CONSTRAINT fk_cc_application_train
FOREIGN KEY (sk_id_curr) REFERENCES application_train(sk_id_curr);

ALTER TABLE credit_card_balance
ADD CONSTRAINT fk_cc_previous_application
FOREIGN KEY (sk_id_prev) REFERENCES previous_application(sk_id_prev);

ALTER TABLE pos_cash_balance
ADD CONSTRAINT fk_pc_application_train
FOREIGN KEY (sk_id_curr) REFERENCES application_train(sk_id_curr);

ALTER TABLE pos_cash_balance
ADD CONSTRAINT fk_pc_previous_application
FOREIGN KEY (sk_id_prev) REFERENCES previous_application(sk_id_prev);